In [10]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

from dotenv import load_dotenv

load_dotenv()

from google import genai
from google.genai import types

ai = genai.Client()

In [6]:
documents = load_faq_data()
index = build_index(documents)


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=ai,
    instructions=instructions,
)

In [7]:
assistant.rag('How do I run Olama locally?')

'Based on the provided context, there is no information on how to run Olama (or Ollama) locally.'

## Asking without tools

In [13]:
question = 'How do I run Olama locally?'
response = ai.models.generate_content(
    model="gemini-3.5-flash",
    contents=[
        types.Content(
            role = "user",
            parts = [
                types.Part.from_text(text = question)
            ]
        )
    ]
)

response.text

'Running **Ollama** (spelled with two "l"s) locally is very straightforward. It allows you to run powerful Large Language Models (LLMs) like Llama 3, Mistral, and Phi-3 directly on your computer without an internet connection.\n\nHere is the step-by-step guide to installing and running Ollama on your machine.\n\n---\n\n### Step 1: Download and Install Ollama\n\nChoose the instructions for your operating system:\n\n#### **For macOS:**\n1. Download the macOS installer from the [official Ollama website](https://ollama.com/download/mac).\n2. Unzip the downloaded file and drag the **Ollama** app into your **Applications** folder.\n3. Open the Ollama app. It will run in your menu bar (top right of your screen).\n\n*(Alternatively, if you use Homebrew, you can install it via terminal: `brew install ollama`)*\n\n#### **For Windows:**\n1. Download the Windows installer from the [official Ollama website](https://ollama.com/download/windows).\n2. Run the `.exe` installer and follow the setup prom

## With function search

In [14]:
index.search(question)

[{'id': 'aa310de435',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Can I run the course locally instead of Codespaces?',
  'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\n\nIf you run locally, make sure you document your setup and keep your environment reproducible.'},
 {'id': '8a23cfbf67',
  'course': 'machine-learning-zoomcamp',
  'section': 'Module 9. Serverless Deep Learning',
  'question': 'How to test AWS Lambda + Docker locally?',
  'answer': 'This deployment setup can be tested locally using [AWS RIE](https://github.com/aws/aws-lambda-runtime-interface-emulator/#test-an-image-with-rie-included-in-the-image) (Runtime Interface Emulator).\n\nIf your Docker image is built upon the base AWS Lambda image (e.g., `FROM public.ecr.aws/lambda/python:3.10`), use specif

In [15]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [16]:
search(question)

[{'id': 'aa310de435',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Can I run the course locally instead of Codespaces?',
  'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\n\nIf you run locally, make sure you document your setup and keep your environment reproducible.'},
 {'id': '193612db63',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Orchestration',
  'question': "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
  'answer': "Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, monitoring, secret management, and reliably chaining tasks together. That's what an orchestrator like Kestra provides.\n\nIn this module Kestra is also the vehicle for the

## Defining tool

In [ ]:
# for openAI
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [27]:
from google.genai.types import GenerateContentConfig, Tool, GoogleSearch
response = ai.models.generate_content(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        tools=[
            Tool(
                google_search=GoogleSearch()
            )
        ],
    ),
    contents=[
        types.Content(
            role = "user",
            parts = [
                types.Part.from_text(text = question)
            ]
        )
    ]
)

response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""Ollama is an open-source platform that enables you to run large language models (LLMs) locally on your own machine, offering benefits like privacy, no API costs, and offline capability. It simplifies the process by packaging model weights, configuration, and necessary software into easy-to-manage bundles.

Before installing, it's worth noting that while Ollama can run models using your CPU and system RAM, performance will be significantly better with a compatible GPU and sufficient VRAM.

Here's a guide on how to install and run Ollama locally based on your operating system:

**1. Install Ollama**

*   **macOS:**
    1.  Visit the official Ollama website (ollama.com) and click "Download," then select "Download for macOS" to get the .zip file.
    2.  Open the downloaded .zip file and drag the Ollama application into y